# Favorite World - male voiceover samples (Claude)
Trained voice vs original Rohit / Aman, 3 takes each, best take picked by Whisper + pitch.

In [ ]:
# CELL 1 - Setup (run every time). Colab or Kaggle.
import os, sys, json, glob, subprocess, getpass
os.environ["USE_TF"] = "0"              # transformers must not load Kaggle's TensorFlow (protobuf clash)
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
SMOKE = False         # True = 15-minute test on a tiny slice (checks everything works before the long run)
ON_KAGGLE = os.path.exists("/kaggle/working")
BATCH = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") == "Batch"     # Kaggle "Save & Run All" / API push
# Private Kaggle dataset with hf_token.txt + favorite_world_voice_dataset.zip (pushed by kaggle_run.py)
INPUT_DIR = next((os.path.dirname(p) for p in glob.glob("/kaggle/input/**/hf_token.txt", recursive=True)), "")
if ON_KAGGLE:
    print("Kaggle inputs:", [p for p in glob.glob("/kaggle/input/**", recursive=True)][:20], "| INPUT_DIR =", INPUT_DIR)
if ON_KAGGLE:
    OUT_ROOT = "/kaggle/working/fw_claude_voice"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = "/content/drive/MyDrive/FavoriteWorld_Voice_Claude"   # Claude's own folder
WORK = "/tmp/fw_claude"                                              # fast local disk (clips, parquet)
PARLER_DIR = os.path.join(WORK, "parler-tts")
for d in (OUT_ROOT, WORK):
    os.makedirs(d, exist_ok=True)

HF_TOKEN = ""   # optional: paste hf_... here, or use Secrets named HF_TOKEN
if not HF_TOKEN and INPUT_DIR:
    HF_TOKEN = open(os.path.join(INPUT_DIR, "hf_token.txt"), encoding="utf-8").read().strip()
if not HF_TOKEN:
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        else:
            from google.colab import userdata
            HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = ""
if not HF_TOKEN and BATCH:
    raise SystemExit("No Hugging Face token found (hf_token.txt input or HF_TOKEN secret).")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your free Hugging Face READ token (hf_...): ").strip()
os.environ["HF_TOKEN"] = HF_TOKEN            # huggingface_hub / datasets read the token from here

!nvidia-smi -L
if not os.path.exists(PARLER_DIR):
    !git clone -q https://github.com/huggingface/parler-tts.git {PARLER_DIR}
!pip install -q -e "{PARLER_DIR}[train]" "datasets==3.6.0" soundfile librosa accelerate gradio gdown faster-whisper
# pip just swapped package versions: forget any copies Python loaded before the install (Kaggle preloads some)
for _m in [m for m in list(sys.modules) if m.split(".")[0] in ("huggingface_hub", "datasets", "transformers", "tokenizers", "accelerate", "fsspec")]:
    del sys.modules[_m]
import importlib; importlib.invalidate_caches()
import huggingface_hub, datasets
print("huggingface_hub", huggingface_hub.__version__, "| datasets", datasets.__version__)
print("OUT_ROOT =", OUT_ROOT, "| WORK =", WORK, "| Kaggle" if ON_KAGGLE else "| Colab", "| SMOKE TEST" if SMOKE else "")

In [ ]:
%%writefile fw_numbers.py
# CELL 2 - number reader
"""fw_numbers.py - read numbers, dates, money, units and short forms the way a Favorite World
narrator says them, in Hindi and in English (Claude, 2026-11-16).

Why: TTS models (Indic-Parler, Chatterbox, Cartesia ...) are trained on spoken words, so digits like
"1947", "₹2.5 लाख", "56.4%", "B12" or "इवान III" come out wrong or get skipped. Real TTS systems fix
this in the text front-end, before the model. This module is that front-end, and the Claude training
notebook runs every training transcript through it too, so training text and generation text match.

Follows production/SCRIPT_STYLE.md "Spell numbers, dates and short forms for the voice":
  years "उन्नीस सौ सैंतालीस", "दो हज़ार तेईस-चौबीस"; decimals "छप्पन दशमलव चार प्रतिशत"; ranges "बीस से पच्चीस";
  codes in English "vitamin B twelve", "COVID nineteen"; acronyms "I-C-M-R"; units in English ("kilometer",
  "kilo"); "%"->प्रतिशत, "₹"->रुपये, "+"->से ज़्यादा, "/"->या, "&"->और; kings' numbers in English
  ("इवान third"); "six", never छह (user, 2026-11-16).

Use:
  from fw_numbers import normalize
  normalize("सन 1947 में ₹2.5 लाख और 56.4% लोग")      # Hindi (auto: any Devanagari -> Hindi)
  normalize("In 1947, 3.5 million people", lang="en")

CLI:
  python production/pipeline/fw_numbers.py "text ..."          # print the spoken form
  python production/pipeline/fw_numbers.py --file lines.txt    # one line per line
  python production/pipeline/fw_numbers.py --selftest
"""
from __future__ import annotations

import re
import sys

# --------------------------------------------------------------------------------------------
# Word tables
# --------------------------------------------------------------------------------------------
HI_0_99 = [
    "शून्य", "एक", "दो", "तीन", "चार", "पाँच", "छह", "सात", "आठ", "नौ",
    "दस", "ग्यारह", "बारह", "तेरह", "चौदह", "पंद्रह", "सोलह", "सत्रह", "अठारह", "उन्नीस",
    "बीस", "इक्कीस", "बाईस", "तेईस", "चौबीस", "पच्चीस", "छब्बीस", "सत्ताईस", "अट्ठाईस", "उनतीस",
    "तीस", "इकतीस", "बत्तीस", "तैंतीस", "चौंतीस", "पैंतीस", "छत्तीस", "सैंतीस", "अड़तीस", "उनतालीस",
    "चालीस", "इकतालीस", "बयालीस", "तैंतालीस", "चवालीस", "पैंतालीस", "छियालीस", "सैंतालीस", "अड़तालीस", "उनचास",
    "पचास", "इक्यावन", "बावन", "तिरपन", "चौवन", "पचपन", "छप्पन", "सत्तावन", "अट्ठावन", "उनसठ",
    "साठ", "इकसठ", "बासठ", "तिरसठ", "चौंसठ", "पैंसठ", "छियासठ", "सड़सठ", "अड़सठ", "उनहत्तर",
    "सत्तर", "इकहत्तर", "बहत्तर", "तिहत्तर", "चौहत्तर", "पचहत्तर", "छिहत्तर", "सतहत्तर", "अठहत्तर", "उन्यासी",
    "अस्सी", "इक्यासी", "बयासी", "तिरासी", "चौरासी", "पचासी", "छियासी", "सत्तासी", "अट्ठासी", "नवासी",
    "नब्बे", "इक्यानवे", "बानवे", "तिरानवे", "चौरानवे", "पचानवे", "छियानवे", "सत्तानवे", "अट्ठानवे", "निन्यानवे",
]
EN_0_19 = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
           "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
EN_TENS = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
EN_ORD_IRREG = {"one": "first", "two": "second", "three": "third", "five": "fifth", "eight": "eighth",
                "nine": "ninth", "twelve": "twelfth"}
HI_MONTHS = ["जनवरी", "फ़रवरी", "मार्च", "अप्रैल", "मई", "जून", "जुलाई", "अगस्त", "सितंबर", "अक्टूबर", "नवंबर", "दिसंबर"]
EN_MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August", "September",
             "October", "November", "December"]
ROMAN = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6, "VII": 7, "VIII": 8, "IX": 9, "X": 10,
         "XI": 11, "XII": 12, "XIII": 13, "XIV": 14, "XV": 15, "XVI": 16, "XVII": 17, "XVIII": 18,
         "XIX": 19, "XX": 20, "XXI": 21}
# Short forms said as a word, not letter by letter.
SAY_AS_WORD = {"NASA", "ISRO", "NATO", "UNESCO", "UNICEF", "AIIMS", "SEBI", "COVID", "OPEC", "SAARC", "BRICS",
               "ASEAN", "NITI", "RAW", "LASER", "RADAR", "SONAR", "ISIS", "NEET", "GATE", "CAT", "AIDS", "SARS",
               "MERS", "QUAD", "FIFA", "UEFA", "OK", "TV", "AI", "SIM", "PIN", "ATM", "GPS", "CCTV", "WHO", "UNO",
               "INTERPOL", "SWIFT", "FASTag", "DOGE", "BHIM", "SWAYAM", "CERN", "IKEA", "NOTA", "ULFA", "AQI"}
# These are not short forms (all-caps emphasis words, Roman numerals) - leave alone.
NOT_ACRONYM = set(ROMAN) | {"A", "MKI", "OR", "AND", "THE", "IN", "ON", "AT", "TO"}

DEV = r"ऀ-ॿ"
_DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")


# --------------------------------------------------------------------------------------------
# Numbers -> words
# --------------------------------------------------------------------------------------------
def hi_int(n: int) -> str:
    """Indian-system Hindi words: 1947 -> उन्नीस सौ सैंतालीस, 250000 -> दो लाख पचास हज़ार."""
    if n < 0:
        return "माइनस " + hi_int(-n)
    if n < 100:
        return HI_0_99[n]
    if 1100 <= n <= 1999:                       # 1947, 1500 -> "उन्नीस सौ सैंतालीस", "पंद्रह सौ"
        c, r = divmod(n, 100)
        return f"{HI_0_99[c]} सौ" + (f" {HI_0_99[r]}" if r else "")
    parts: list[str] = []
    if n >= 10_000_000:
        parts.append(hi_int(n // 10_000_000) + " करोड़")
        n %= 10_000_000
    if n >= 100_000:
        parts.append(HI_0_99[n // 100_000] + " लाख")
        n %= 100_000
    if n >= 1_000:
        parts.append(HI_0_99[n // 1_000] + " हज़ार")
        n %= 1_000
    if n >= 100:
        parts.append(HI_0_99[n // 100] + " सौ")
        n %= 100
    if n:
        parts.append(HI_0_99[n])
    return " ".join(parts)


def en_int(n: int) -> str:
    if n < 0:
        return "minus " + en_int(-n)
    if n < 20:
        return EN_0_19[n]
    if n < 100:
        t, o = divmod(n, 10)
        return EN_TENS[t] + (f"-{EN_0_19[o]}" if o else "")
    if n < 1000:
        h, r = divmod(n, 100)
        return f"{EN_0_19[h]} hundred" + (f" {en_int(r)}" if r else "")
    for div, name in ((10**12, "trillion"), (10**9, "billion"), (10**6, "million"), (10**3, "thousand")):
        if n >= div:
            q, r = divmod(n, div)
            return f"{en_int(q)} {name}" + (f" {en_int(r)}" if r else "")
    return str(n)


def en_year(n: int) -> str:
    if 2000 <= n <= 2009:
        return en_int(n)
    if 1000 <= n <= 2099:
        hi, lo = divmod(n, 100)
        if lo == 0:
            return f"{en_int(hi)} hundred"
        return f"{en_int(hi)} " + (f"oh {EN_0_19[lo]}" if lo < 10 else en_int(lo))
    return en_int(n)


def en_ordinal(n: int) -> str:
    w = en_int(n)
    head, sep, last = w.rpartition("-") if "-" in w.split(" ")[-1] else w.rpartition(" ")
    if last in EN_ORD_IRREG:
        last = EN_ORD_IRREG[last]
    elif last.endswith("y"):
        last = last[:-1] + "ieth"
    else:
        last += "th"
    return head + sep + last


def hi_decimal(left: str, right: str) -> str:
    li = int(left or "0")
    if right == "5" and 1 <= li <= 999:          # 1.5 डेढ़, 2.5 ढाई, 3.5 साढ़े तीन
        if li == 1:
            return "डेढ़"
        if li == 2:
            return "ढाई"
        return "साढ़े " + hi_int(li)
    return f"{hi_int(li)} दशमलव " + " ".join(HI_0_99[int(d)] for d in right)


def en_decimal(left: str, right: str) -> str:
    return f"{en_int(int(left or '0'))} point " + " ".join(EN_0_19[int(d)] for d in right)


def num_words(raw: str, lang: str, year_ok: bool = True) -> str:
    """'1,23,456' / '56.4' / '1947' -> words."""
    s = raw.replace(",", "")
    if "." in s:
        left, right = s.split(".", 1)
        return hi_decimal(left, right) if lang == "hi" else en_decimal(left, right)
    n = int(s)
    if lang == "hi":
        return hi_int(n)
    if year_ok and "," not in raw and len(s) == 4 and 1100 <= n <= 2099:
        return en_year(n)
    return en_int(n)


def digits_one_by_one(s: str, lang: str) -> str:
    table = HI_0_99 if lang == "hi" else EN_0_19
    return " ".join(table[int(d)] for d in s if d.isdigit())


# --------------------------------------------------------------------------------------------
# Text passes
# --------------------------------------------------------------------------------------------
NUM = r"\d{1,3}(?:,\d{2,3})+(?:\.\d+)?|\d+(?:\.\d+)?"
MAG_HI = r"हज़ार|हजार|लाख|करोड़|करोड|अरब|खरब"
MAG_EN = r"thousand|lakh|lakhs|crore|crores|million|billion|trillion|mn|bn|cr|k|m|b"
UNITS = [  # (regex after a number, Hindi word, English word)
    (r"km/h|kmph|km/hr", "kilometer per hour", "kilometers per hour"),
    (r"mph", "miles per hour", "miles per hour"),
    (r"sq\.?\s?km|km²|km2", "square kilometer", "square kilometers"),
    (r"km", "kilometer", "kilometers"), (r"cm", "centimeter", "centimeters"),
    (r"mm", "millimeter", "millimeters"), (r"kg", "kilo", "kilograms"), (r"mg", "milligram", "milligrams"),
    (r"g", "gram", "grams"), (r"ml", "milliliter", "milliliters"), (r"m", "meter", "meters"),
    (r"ft", "feet", "feet"), (r"°\s?C", "degree Celsius", "degrees Celsius"),
    (r"°\s?F", "degree Fahrenheit", "degrees Fahrenheit"), (r"°", "degree", "degrees"),
    (r"kWh", "kilowatt hour", "kilowatt hours"), (r"kW", "kilowatt", "kilowatts"),
    (r"MW", "megawatt", "megawatts"), (r"GW", "gigawatt", "gigawatts"),
    (r"GB", "GB", "GB"), (r"TB", "TB", "TB"), (r"MB", "MB", "MB"),
    (r"Mbps", "Mbps", "Mbps"), (r"GHz", "gigahertz", "gigahertz"), (r"MHz", "megahertz", "megahertz"),
]
CURRENCY = [  # (regex, Hindi, English singular, English plural)
    (r"₹|Rs\.?|INR", "रुपये", "rupee", "rupees"), (r"\$|USD|US\$", "डॉलर", "dollar", "dollars"),
    (r"€|EUR", "यूरो", "euro", "euros"), (r"£|GBP", "पाउंड", "pound", "pounds"),
    (r"¥", "येन", "yen", "yen"),
]
_MAG_MAP_HI = {"thousand": "हज़ार", "k": "हज़ार", "lakh": "लाख", "lakhs": "लाख", "crore": "करोड़", "crores": "करोड़",
               "cr": "करोड़", "million": "million", "mn": "million", "m": "million", "billion": "billion",
               "bn": "billion", "b": "billion", "trillion": "trillion", "हजार": "हज़ार", "करोड": "करोड़"}
_MAG_MAP_EN = {"k": "thousand", "mn": "million", "m": "million", "bn": "billion", "b": "billion", "cr": "crore",
               "lakhs": "lakh", "crores": "crore", "हज़ार": "thousand", "हजार": "thousand", "लाख": "lakh",
               "करोड़": "crore", "करोड": "crore", "अरब": "billion"}


def _mag(word: str, lang: str) -> str:
    w = word.strip()
    key = w.lower() if re.match(r"[A-Za-z]", w) else w
    return (_MAG_MAP_HI if lang == "hi" else _MAG_MAP_EN).get(key, w)


def _sp(s: str) -> str:
    return f" {s} "


def _pass_phones(s: str, lang: str) -> str:
    def phone(m: re.Match) -> str:
        body = m.group(0)
        plus = body.strip().startswith("+")
        words = digits_one_by_one(body, lang)
        return _sp(("plus " if plus else "") + words)
    s = re.sub(r"\+\d{1,3}[\s-]?\d{3,5}[\s-]?\d{3,6}(?:[\s-]?\d{2,5})?", phone, s)
    s = re.sub(r"(?<![\d.,])(?:0\d{2,}|\d{10,})(?![\d.,])", phone, s)  # 0xxxx codes, 10+ digit strings
    return s


def _pass_dates(s: str, lang: str) -> str:
    def date(m: re.Match) -> str:
        d, mo, y = int(m.group(1)), int(m.group(2)), m.group(3)
        if not (1 <= d <= 31 and 1 <= mo <= 12):
            return m.group(0)
        yy = int(y) if len(y) == 4 else 2000 + int(y)
        if lang == "hi":
            return _sp(f"{hi_int(d)} {HI_MONTHS[mo - 1]} {hi_int(yy)}")
        return _sp(f"{en_ordinal(d)} {EN_MONTHS[mo - 1]} {en_year(yy)}")
    s = re.sub(r"(?<![\d.])(\d{1,2})[/.-](\d{1,2})[/.-](\d{4}|\d{2})(?![\d.])", date, s)

    def day_month(m: re.Match) -> str:           # 26/11 -> छब्बीस ग्यारह, 9/11 -> nine eleven
        a, b = int(m.group(1)), int(m.group(2))
        if m.group(0) == "9/11":
            return _sp("nine eleven")
        return _sp(f"{hi_int(a)} {hi_int(b)}" if lang == "hi" else f"{en_int(a)} {en_int(b)}")
    return re.sub(r"(?<![\d./])(\d{1,2})/(\d{1,2})(?![\d/])", day_month, s)


def _pass_times(s: str, lang: str) -> str:
    def tm(m: re.Match) -> str:
        h, mi = int(m.group(1)), int(m.group(2))
        if h > 24 or mi > 59:
            return m.group(0)
        ampm = (m.group(3) or "").strip()
        if lang == "hi":
            nxt = 1 if h == 12 else h + 1
            if mi == 0:
                out = f"{hi_int(h)} बजे"
            elif mi == 30:
                out = "डेढ़ बजे" if h == 1 else "ढाई बजे" if h == 2 else f"साढ़े {hi_int(h)} बजे"
            elif mi == 15:
                out = f"सवा {hi_int(h)} बजे"
            elif mi == 45:
                out = f"पौने {hi_int(nxt)} बजे"
            else:
                out = f"{hi_int(h)} बजकर {hi_int(mi)} मिनट"
            if ampm:                              # 10:30 PM -> रात साढ़े दस बजे
                pm = ampm.lower().startswith("p")
                h24 = (h % 12) + (12 if pm else 0)
                part = "सुबह" if h24 < 12 else "दोपहर" if h24 < 16 else "शाम" if h24 < 19 else "रात"
                if h24 < 4:
                    part = "रात"
                return _sp(f"{part} {out}")
        else:
            out = f"{en_int(h)} o'clock" if mi == 0 and not ampm else (
                f"{en_int(h)}" + ("" if mi == 0 else f" {('oh ' + EN_0_19[mi]) if mi < 10 else en_int(mi)}"))
        return _sp(out + (f" {ampm.upper().replace('.', '')}" if ampm else ""))
    s = re.sub(r"(?<![\d:])(\d{1,2}):(\d{2})(?![\d:])(\s?(?:am|pm|AM|PM|a\.m\.|p\.m\.))?", tm, s)
    return re.sub(r"(बजे|मिनट)\s+बजे", r"\1", s)       # "10:30 बजे" -> "साढ़े दस बजे" (no double बजे)


def _pass_currency(s: str, lang: str) -> str:
    for rx, hi_w, en_1, en_n in CURRENCY:
        pat = (rf"(?:{rx})\s?({NUM})(?:\s?({MAG_HI}|{MAG_EN})(?![A-Za-z{DEV}]))?"
               rf"(?:\s({MAG_HI}|crore|crores)(?![A-Za-z{DEV}]))?"
               rf"(?:\s(\d{{1,2}})\s({MAG_HI}|thousand|lakh)(?![A-Za-z{DEV}]))?")   # ₹1 लाख 20 हज़ार

        def cur(m: re.Match, hi_w=hi_w, en_1=en_1, en_n=en_n) -> str:
            n = num_words(m.group(1), lang, year_ok=False)
            mags = [ _mag(g, lang) for g in (m.group(2), m.group(3)) if g ]
            if m.group(4):
                mags.append(num_words(m.group(4), lang) + " " + _mag(m.group(5), lang))
            tail = " ".join(mags)
            if lang == "hi":
                return _sp(" ".join(x for x in (n, tail, hi_w) if x))
            one = m.group(1) in ("1", "1.0") and not mags
            return _sp(" ".join(x for x in (n, tail, en_1 if one else en_n) if x))
        s = re.sub(pat, cur, s)
    return s


def _pass_ranges(s: str, lang: str) -> str:
    to = "से" if lang == "hi" else "to"

    def yr_short(m: re.Match) -> str:            # 2023-24 -> दो हज़ार तेईस-चौबीस
        a, b = int(m.group(1)), int(m.group(2))
        if lang == "hi":
            return _sp(f"{hi_int(a)}-{hi_int(b)}")
        return _sp(f"{en_year(a)} {to} {en_int(b)}")
    s = re.sub(r"(?<![\d.])(1[1-9]\d\d|20\d\d)\s?[-–]\s?(\d\d)(?![\d.%])", yr_short, s)

    def rng(m: re.Match) -> str:
        a, b, pct = m.group(1), m.group(2), m.group(3)
        out = f"{num_words(a, lang)} {to} {num_words(b, lang)}"
        if pct:
            out += " प्रतिशत" if lang == "hi" else " percent"
        return _sp(out)
    return re.sub(rf"(?<![\d.\w-])({NUM})\s?[-–]\s?({NUM})(%)?(?![\d.])", rng, s)


def _pass_ordinals(s: str, lang: str) -> str:
    return re.sub(r"(?<![\d.])(\d+)(st|nd|rd|th)\b", lambda m: _sp(en_ordinal(int(m.group(1)))), s)


def _pass_roman(s: str, lang: str) -> str:
    def rom(m: re.Match) -> str:
        name, r = m.group(1), m.group(2)
        if r == "I" and not re.search(rf"[{DEV}]", name):
            return m.group(0)                     # English "I" pronoun - leave
        n = ROMAN[r]
        if name.lower() in ("war", "युद्ध", "वॉर"):
            return f"{name} {en_int(n)}"          # World War two
        return f"{name} {en_ordinal(n)}"          # इवान third, Henry eighth
    keys = "|".join(sorted(ROMAN, key=len, reverse=True))
    return re.sub(rf"([A-Z][a-z]+|[{DEV}]+)\s+({keys})(?![A-Za-z{DEV}\d])", rom, s)


def _pass_en_months(s: str, lang: str) -> str:
    mon = "|".join(EN_MONTHS)
    s = re.sub(rf"\b({mon})\s+(\d{{1,2}})(?!\d)(?:,\s?(\d{{4}}))?",
               lambda m: f"{m.group(1)} {en_ordinal(int(m.group(2)))}" + (f", {en_year(int(m.group(3)))}" if m.group(3) else ""), s)
    s = re.sub(rf"(?<!\d)(\d{{1,2}})\s+({mon})\b", lambda m: f"{en_ordinal(int(m.group(1)))} {m.group(2)}", s)
    return s


def _spell_acronym(word: str) -> str:
    if word in SAY_AS_WORD or word in NOT_ACRONYM or len(word) > 6:
        return word
    return "-".join(word)


def _pass_codes(s: str, lang: str) -> str:
    """B12 -> B twelve, COVID-19 -> COVID nineteen, F-16 -> F sixteen, 5G -> five G (always English)."""
    def code(m: re.Match) -> str:
        letters, num, after = m.group(1), m.group(2), m.group(3) or ""
        lw = _spell_acronym(letters) if letters.isupper() else letters
        return f"{lw} {code_num(num)}" + (f" {_spell_acronym(after)}" if after else "")
    s = re.sub(r"(?<![A-Za-z\d])([A-Za-z]{1,15})-?(\d{1,4})([A-Z]{1,4})?(?![\d.]|[a-z])", code, s)
    s = re.sub(rf"([{DEV}])-(?=\d)", r"\1 ", s)            # चंद्रयान-3 -> चंद्रयान 3
    s = re.sub(r"(?<![\w.])(\d{1,2})([A-Z])(?![A-Za-z\d])", lambda m: f"{en_int(int(m.group(1)))} {m.group(2)}", s)
    return s


def code_num(num: str) -> str:
    """Model/code numbers the way people say them: 320 -> three twenty, 737 -> seven thirty-seven."""
    n = int(num)
    if len(num) == 3 and n % 100 and n >= 100:
        lo = n % 100
        return f"{EN_0_19[n // 100]} " + (f"oh {EN_0_19[lo]}" if lo < 10 else en_int(lo))
    if len(num) == 4:
        return en_year(n)
    return en_int(n)


def _pass_decades(s: str, lang: str) -> str:
    """1990s -> उन्नीस सौ नब्बे के दशक / nineteen nineties."""
    def dec(m: re.Match) -> str:
        n = int(m.group(1))
        if lang == "hi":
            return _sp(f"{hi_int(n)} के दशक")
        w = en_year(n) if n >= 1000 else en_int(n)
        head, _, last = w.rpartition(" ") if " " in w else ("", "", w)
        last = last[:-1] + "ies" if last.endswith("y") else last + "s"
        return _sp((head + " " if head else "") + last)
    return re.sub(r"(?<![\d.])(\d{2,3}0)'?s\b", dec, s)


def _pass_times_by(s: str, lang: str) -> str:
    """24x7 -> twenty-four seven; 4x4 -> four by four (said in English)."""
    def by(m: re.Match) -> str:
        a, b = int(m.group(1)), int(m.group(2))
        return _sp(f"{en_int(a)} seven" if (a, b) == (24, 7) else f"{en_int(a)} by {en_int(b)}")
    return re.sub(r"(?<![\d.])(\d{1,3})\s?[x×]\s?(\d{1,3})(?![\d.])", by, s)


def _pass_units(s: str, lang: str) -> str:
    for rx, hi_w, en_w in UNITS:
        word = hi_w if lang == "hi" else en_w
        s = re.sub(rf"({NUM})\s?(?:{rx})(?![A-Za-z²\d])", lambda m, w=word: f"{m.group(1)} {w}", s)
    return s


def _pass_symbols_near_numbers(s: str, lang: str) -> str:
    hi = lang == "hi"
    s = re.sub(rf"({NUM})\s?%", lambda m: f"{m.group(1)} {'प्रतिशत' if hi else 'percent'}", s)
    s = re.sub(rf"({NUM})\s?\+(?![\d])", lambda m: f"{m.group(1)} {'से ज़्यादा' if hi else 'plus'}", s)
    s = re.sub(rf"~\s?({NUM})", lambda m: f"{'लगभग' if hi else 'about'} {m.group(1)}", s)
    s = re.sub(rf"(?<![A-Za-z])({NUM})\s?[x×](?![A-Za-z\d])", lambda m: f"{m.group(1)} {'गुना' if hi else 'times'}", s)
    s = re.sub(rf"(?<![\w\d])#\s?({NUM})", lambda m: f"{'नंबर' if hi else 'number'} {m.group(1)}", s)
    s = re.sub(rf"(?:(?<=\s)|^)[-−]({NUM})", lambda m: f"{'माइनस' if hi else 'minus'} {m.group(1)}", s)
    return s


def _pass_mag_suffix(s: str, lang: str) -> str:
    """10k / 5cr / 2bn written without currency -> दस हज़ार / five crore."""
    return re.sub(r"(?<![\w.])(\d+(?:\.\d+)?)\s?(k|cr|bn|mn)(?![A-Za-z])",
                  lambda m: f"{m.group(1)} {_mag(m.group(2), lang)}", s)


def _pass_plain_numbers(s: str, lang: str) -> str:
    return re.sub(rf"(?<![\d.])({NUM})(?![\d])", lambda m: _sp(num_words(m.group(1), lang)), s)


def _pass_acronyms(s: str, lang: str) -> str:
    return re.sub(r"(?<![A-Za-z])([A-Z]{2,6})(?![A-Za-z])", lambda m: _spell_acronym(m.group(1)), s)


def _pass_symbols(s: str, lang: str) -> str:
    hi = lang == "hi"
    s = re.sub(r"\s&\s|(?<=[A-Za-z])&(?=[A-Za-z])", " और " if hi else " and ", s)
    s = re.sub(rf"(?<=[A-Za-z{DEV}])\s?/\s?(?=[A-Za-z{DEV}])", " या " if hi else " or ", s)
    return s


def _six_rule(s: str) -> str:
    """User rule 2026-11-16: 'instead of छह must write six'."""
    return re.sub(rf"(?<![{DEV}\w])(?:छह|छः|छे)(?![{DEV}\w])", "six", s)


def detect_lang(text: str) -> str:
    return "hi" if re.search(rf"[{DEV}]", text or "") else "en"


def normalize(text: str, lang: str = "auto", six_english: bool = True, spell_acronyms: bool = True) -> str:
    """Return the text with every number, date, money amount, unit and short form written as spoken words."""
    s = (text or "").translate(_DEV_DIGITS)
    if lang == "auto":
        lang = detect_lang(s)
    s = s.replace("−", "-").replace("–", "-").replace(" ", " ")
    for fn in (_pass_phones, _pass_dates, _pass_times, _pass_currency, _pass_decades, _pass_times_by,
               _pass_ranges, _pass_ordinals,
               _pass_roman, _pass_en_months, _pass_units, _pass_codes, _pass_symbols_near_numbers,
               _pass_mag_suffix, _pass_plain_numbers):
        s = fn(s, lang)
    if spell_acronyms:
        s = _pass_acronyms(s, lang)
    s = _pass_symbols(s, lang)
    if lang == "hi" and six_english:
        s = _six_rule(s)
    s = re.sub(r"\s+([,।.?!:;])", r"\1", s)
    s = re.sub(r"\(\s+", "(", s)
    s = re.sub(r"\s+\)", ")", s)
    return re.sub(r"\s{2,}", " ", s).strip()


def has_number(text: str) -> bool:
    """True if a line has digits or spoken number words (used to pick number-rich training clips)."""
    t = text or ""
    if re.search(r"[\d०-९]", t):
        return True
    # "एक"/"दो" (a / give) and "one"/"first"/"second" are everyday words - they don't make a line number-rich.
    words = (set(HI_0_99[3:]) | {"सौ", "हज़ार", "हजार", "लाख", "करोड़", "दशमलव", "प्रतिशत", "डेढ़", "ढाई", "साढ़े",
                                  "सवा", "पौने", "अरब"}) - {"नौ"}
    toks = re.findall(rf"[{DEV}]+", t)
    if any(w in words for w in toks):
        return True
    en = (set(EN_0_19[3:]) | set(EN_TENS[2:]) | {"hundred", "thousand", "million", "billion", "lakh", "crore",
                                                  "percent", "third"}) - {"second"}
    return any(w in en for w in re.findall(r"[a-z]+", t.lower()))


# --------------------------------------------------------------------------------------------
# Self-test
# --------------------------------------------------------------------------------------------
TESTS = [
    ("सन 1947 में भारत आज़ाद हुआ।", "सन उन्नीस सौ सैंतालीस में भारत आज़ाद हुआ।"),
    ("2023-24 में 56.4% लोग।", "दो हज़ार तेईस-चौबीस में छप्पन दशमलव चार प्रतिशत लोग।"),
    ("₹2.5 लाख करोड़ का बजट", "ढाई लाख करोड़ रुपये का बजट"),
    ("₹500 का नोट", "पाँच सौ रुपये का नोट"),
    ("$10 million का सौदा", "दस million डॉलर का सौदा"),
    ("20-25 किलोमीटर", "बीस से पच्चीस किलोमीटर"),
    ("10-15% बढ़त", "दस से पंद्रह प्रतिशत बढ़त"),
    ("vitamin B12 की कमी", "vitamin B twelve की कमी"),
    ("COVID-19 के बाद ICMR ने कहा", "COVID nineteen के बाद I-C-M-R ने कहा"),
    ("NFHS-5 का डेटा", "N-F-H-S five का डेटा"),
    ("इवान III और निकोलस II", "इवान third और निकोलस second"),
    ("विश्व युद्ध II में", "विश्व युद्ध two में"),
    ("6 लोग और 600 सैनिक", "six लोग और six सौ सैनिक"),
    ("1,00,000 लोग", "एक लाख लोग"),
    ("15/08/1947 को", "पंद्रह अगस्त उन्नीस सौ सैंतालीस को"),
    ("सुबह 10:30 बजे", "सुबह साढ़े दस बजे"),
    ("शाम 6:45 और 7:20 पर", "शाम पौने सात बजे और सात बजकर बीस मिनट पर"),
    ("1990s में", "उन्नीस सौ नब्बे के दशक में"),
    ("The 1990s were wild", "The nineteen nineties were wild"),
    ("मुंबई 26/11 और 9/11 हमला", "मुंबई छब्बीस ग्यारह और nine eleven हमला"),
    ("ISRO के Chandrayaan-3 और PSLV-C57, चंद्रयान-3", "ISRO के Chandrayaan three और P-S-L-V-C fifty-seven, चंद्रयान तीन"),
    ("A320 और Boeing 737", "A three twenty और Boeing सात सौ सैंतीस"),
    ("24x7 सेवा", "twenty-four seven सेवा"),
    ("₹1 लाख 20 हज़ार की चोरी", "एक लाख बीस हज़ार रुपये की चोरी"),
    ("10:30 PM पर", "रात साढ़े दस बजे पर"),
    ("1971 की जंग में 93,000 सैनिक", "उन्नीस सौ इकहत्तर की जंग में तिरानवे हज़ार सैनिक"),
    ("PM मोदी और AK-47", "P-M मोदी और A-K forty-seven"),
    ("100 km/h की रफ़्तार, 50 km दूर, 5 kg सोना", "एक सौ kilometer per hour की रफ़्तार, पचास kilometer दूर, पाँच kilo सोना"),
    ("500+ लोग", "पाँच सौ से ज़्यादा लोग"),
    ("-5°C तापमान", "माइनस पाँच degree Celsius तापमान"),
    ("5G और 4K", "five G और four K"),
    ("१९९१ में", "उन्नीस सौ इक्यानवे में"),
    ("3.5 घंटे", "साढ़े तीन घंटे"),
    ("1.5 करोड़", "डेढ़ करोड़"),
    ("1st रैंक", "first रैंक"),
    ("F-16 और MiG-21", "F sixteen और MiG twenty-one"),
    ("soy/tofu खाइए", "soy या tofu खाइए"),
    ("In 1947, 3.5 million people moved.", "In nineteen forty-seven, three point five million people moved."),
    ("It cost $1 and ₹2.5 crore.", "It cost one dollar and two point five crore rupees."),
    ("On August 15, 1947 India", "On August fifteenth, nineteen forty-seven India"),
    ("Henry VIII had 6 wives", "Henry eighth had six wives"),
    ("Growth was 7.2% in 2005 and 2023.", "Growth was seven point two percent in two thousand five and twenty twenty-three."),
    ("Call 9876543210 now", "Call nine eight seven six five four three two one zero now"),
    ("The 21st century", "The twenty-first century"),
    ("Then I went", "Then I went"),
    ("He scored 1,234 runs", "He scored one thousand two hundred thirty-four runs"),
    ("10k लोग", "दस हज़ार लोग"),
]


def selftest() -> int:
    bad = 0
    for src, want in TESTS:
        got = normalize(src)
        ok = got == want
        bad += not ok
        print(("PASS " if ok else "FAIL ") + src + "\n     -> " + got + ("" if ok else "\n   want " + want))
    print(f"\n{len(TESTS) - bad}/{len(TESTS)} passed")
    return 1 if bad else 0


def main(argv: list[str]) -> int:
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    if not argv or argv[0] in ("-h", "--help"):
        print(__doc__)
        return 0
    if argv[0] == "--selftest":
        return selftest()
    lang = "auto"
    if "--lang" in argv:
        i = argv.index("--lang")
        lang = argv[i + 1]
        argv = argv[:i] + argv[i + 2:]
    if argv[0] == "--file":
        with open(argv[1], encoding="utf-8") as f:
            for line in f:
                print(normalize(line.rstrip("\n"), lang=lang))
        return 0
    print(normalize(" ".join(argv), lang=lang))
    return 0


if __name__ == "__main__":
    raise SystemExit(main(sys.argv[1:]))


In [ ]:
VOICE_MODULE = '"""fw_voice.py - load the trained voice and speak (written by the notebook)."""\nimport os, sys, re, json, glob, gc\nos.environ.setdefault("USE_TF", "0")\nos.environ.setdefault("TRANSFORMERS_NO_TF", "1")\nimport numpy as np, torch, soundfile as sf\nOUT_ROOT = os.environ["FW_OUT_ROOT"]\nPARLER_DIR = os.environ["FW_PARLER_DIR"]\nSMOKE = os.environ.get("FW_SMOKE") == "1"\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nsys.path.insert(0, PARLER_DIR)\nimport fw_numbers as fwn\nfrom parler_tts import ParlerTTSForConditionalGeneration\nfrom transformers import AutoTokenizer\n\nMODEL_OUT = os.path.join(OUT_ROOT, "model")\nFW_DESC = ("FavoriteWorld speaks clearly in a warm, expressive, studio-quality Hindi documentary "\n           "narrator voice with natural pacing and no background noise.")\nDEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"\n\n\ndef load_voice(which="trained"):\n    """trained = final model, or the latest checkpoint if training stopped early; base = original indic-parler-tts."""\n    base = "ai4bharat/indic-parler-tts"\n    if which == "trained" and os.path.exists(os.path.join(MODEL_OUT, "config.json")):\n        model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_OUT)\n        tok_dir = MODEL_OUT\n    else:\n        model = ParlerTTSForConditionalGeneration.from_pretrained(base)\n        tok_dir = base\n        if which == "trained":\n            ck = sorted(glob.glob(os.path.join(MODEL_OUT, "checkpoint-*")), key=lambda p: int(re.findall(r"checkpoint-(\\d+)", p)[0]))\n            if not ck:\n                raise SystemExit("No trained model or checkpoint yet.")\n            wf = next((p for p in (os.path.join(ck[-1], n) for n in ("pytorch_model.bin", "model.safetensors")) if os.path.exists(p)), None)\n            if wf.endswith(".bin"):\n                sd = torch.load(wf, map_location="cpu")\n            else:\n                from safetensors.torch import load_file\n                sd = load_file(wf)\n            print("Loaded checkpoint", ck[-1], model.load_state_dict(sd, strict=False))\n    model = model.to(DEVICE).eval()\n    if DEVICE.startswith("cuda"):\n        model = model.half()\n    tok = AutoTokenizer.from_pretrained(tok_dir)\n    desc_tok = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)\n    return model, tok, desc_tok\n\n\ndef chunks(text, limit=180):\n    parts = re.split(r"(?<=[।.!?])\\s+", text.strip())\n    out = []\n    for p in parts:\n        while len(p) > limit:\n            cut = p.rfind(",", 0, limit)\n            cut = cut if cut > 40 else limit\n            out.append(p[:cut + 1].strip())\n            p = p[cut + 1:].strip()\n        if p:\n            out.append(p)\n    return out\n\n\ndef speak(voice, text, desc=FW_DESC, normalize=True):\n    model, tok, desc_tok = voice\n    spoken = fwn.normalize(text) if normalize else text\n    sr = model.config.sampling_rate\n    pieces = []\n    d = desc_tok(desc, return_tensors="pt").to(DEVICE)\n    for c in chunks(spoken):\n        p = tok(c, return_tensors="pt").to(DEVICE)\n        with torch.no_grad():\n            g = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,\n                               prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)\n        pieces += [g.float().cpu().numpy().squeeze(), np.zeros(int(sr * 0.12), dtype=np.float32)]\n    return (np.concatenate(pieces) if pieces else np.zeros(1, dtype=np.float32)), sr, spoken\n'
MALE_SCRIPT = '"""fw_male.py - male voiceover takes, best one picked by Whisper + pitch."""\nfrom fw_voice import *\nimport shutil, librosa\nfrom faster_whisper import WhisperModel\n\nTEXT = ("सन 1971 की सर्दियों में, ढाका में 93,000 पाकिस्तानी सैनिकों ने हथियार डाल दिए। "\n        "सिर्फ़ 13 दिन की जंग ने, एशिया का नक्शा बदल दिया।")\nVOICES = {   # name: (model, description)\n    "trained_fw_male": ("trained", "FavoriteWorld speaks in a deep, warm, confident male Hindi documentary narrator voice "\n                                   "with natural pacing, very clear audio and no background noise."),\n    "original_rohit_male": ("base", "Rohit speaks in a deep, confident male voice at a moderate pace, like a documentary "\n                                    "narrator, with very clear, close-up audio and no background noise."),\n    "original_aman_male": ("base", "Aman speaks in a deep, confident male voice at a moderate pace, like a documentary "\n                                   "narrator, with very clear, close-up audio and no background noise."),\n}\nTAKES = 3\nOUT = "/kaggle/working/male"\nos.makedirs(os.path.join(OUT, "takes"), exist_ok=True)\n\n\ndef load16k(path):\n    a, sr0 = sf.read(path, dtype="float32")\n    if a.ndim > 1:\n        a = a.mean(axis=1)\n    return librosa.resample(a, orig_sr=sr0, target_sr=16000).astype(np.float32)\n\n\ndef words(t):\n    t = fwn.normalize(t)\n    return re.sub(r"[^\\wऀ-ॿ\\s]", " ", t.lower()).split()\n\n\ndef cer(ref, hyp):\n    a, b = "".join(words(ref)), "".join(words(hyp))\n    prev = list(range(len(b) + 1))\n    for i, ca in enumerate(a, 1):\n        cur = [i]\n        for j, cb in enumerate(b, 1):\n            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))\n        prev = cur\n    return prev[-1] / max(len(a), 1)\n\n\ndef pitch(path):\n    y, sr0 = librosa.load(path, sr=16000)\n    f0, voiced, _ = librosa.pyin(y, fmin=60, fmax=400, sr=16000)\n    f0 = f0[~np.isnan(f0)]\n    return float(np.median(f0)) if f0.size else 0.0\n\n\ntakes = []\nfor kind in ("trained", "base"):\n    names = [n for n, (k, _) in VOICES.items() if k == kind]\n    if not names:\n        continue\n    voice = load_voice(kind)\n    for name in names:\n        for t in range(1, TAKES + 1):\n            torch.manual_seed(1000 + t)\n            wav, sr, spoken = speak(voice, TEXT, desc=VOICES[name][1])\n            p = os.path.join(OUT, "takes", f"{name}_take{t}.wav")\n            sf.write(p, wav, sr)\n            takes.append({"name": name, "take": t, "path": p, "seconds": round(len(wav) / sr, 1)})\n            print(name, "take", t, takes[-1]["seconds"], "s")\n    del voice; gc.collect(); torch.cuda.empty_cache()\n\ntry:\n    asr = WhisperModel("large-v3", device="cuda", compute_type="float16")\nexcept Exception as e:\n    print("Whisper GPU failed:", e, "- CPU")\n    asr = WhisperModel("large-v3", device="cpu", compute_type="int8")\nfor tk in takes:\n    segs, _ = asr.transcribe(load16k(tk["path"]), language="hi", beam_size=5)\n    tk["heard"] = " ".join(s.text for s in segs).strip()\n    tk["cer"] = round(cer(TEXT, tk["heard"]), 3)\n    tk["pitch_hz"] = round(pitch(tk["path"]), 1)\n    # lower is better: reading errors, plus a penalty if it doesn\'t sound male (median pitch above ~165 Hz)\n    # or falls outside 10-20 s\n    tk["score"] = round(tk["cer"] + (0.5 if tk["pitch_hz"] > 165 else 0) + (0.3 if not 10 <= tk["seconds"] <= 20 else 0), 3)\n    print(tk["name"], tk["take"], "cer", tk["cer"], "pitch", tk["pitch_hz"], "Hz", tk["seconds"], "s |", tk["heard"])\n\nbest = {}\nfor tk in takes:\n    if tk["name"] not in best or tk["score"] < best[tk["name"]]["score"]:\n        best[tk["name"]] = tk\nfor name, tk in best.items():\n    shutil.copy(tk["path"], os.path.join(OUT, name + ".wav"))\n    print(f"BEST {name}: take {tk[\'take\']}, {tk[\'seconds\']} s, pitch {tk[\'pitch_hz\']} Hz, cer {tk[\'cer\']}")\njson.dump({"text": TEXT, "spoken": fwn.normalize(TEXT), "best": best, "takes": takes},\n          open(os.path.join(OUT, "report.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)\n'
# CELL 3 - Find the trained model (output of the notebook fw-voice-claude-train, attached read-only)
import glob
cands = [p for p in glob.glob("/kaggle/input/**/fw_claude_voice/model/config.json", recursive=True)]
print("trained model:", cands)
if not cands:
    raise SystemExit("Trained model not found in the attached notebook output.")
TRAINED_ROOT = os.path.dirname(os.path.dirname(cands[0]))      # .../fw_claude_voice
open("fw_voice.py", "w", encoding="utf-8").write(VOICE_MODULE)
open("fw_male.py", "w", encoding="utf-8").write(MALE_SCRIPT)
SMOKE_FLAG = "0"

In [ ]:
# CELL 4 - Make the male takes in a fresh Python and pick the best
!USE_TF=0 TRANSFORMERS_NO_TF=1 FW_OUT_ROOT="{TRAINED_ROOT}" FW_PARLER_DIR="{PARLER_DIR}" FW_SMOKE=0 python fw_male.py 2>&1 | tee /kaggle/working/male.log
print(sorted(glob.glob("/kaggle/working/male/*.wav")))